In [1]:
"""
Bond Floor Calculator
----------------------
Calculates the straight bond floor for each convertible bond
on every trading day in its history.

Bond Floor = PV of all future coupon payments + PV of face value
           = sum[ C / (1+r_t)^t ] + [ Par / (1+r_T)^T ]

Where:
    C     = semi-annual coupon payment
    r_t   = interpolated discount rate (ZC Treasury + BBB spread)
             matched to each cash flow's exact time to payment
    t     = time to each coupon in years (30/360 day count)
    T     = time to maturity in years
    Par   = 1000 (face value)

Key decisions:
    - Day count: 30/360 (matches Bloomberg field)
    - Coupon frequency: Semi-annual (S/A)
    - Discount rate: Zero coupon Treasury + BBB spread (risk_free + spread)
    - Call provisions: Ignored — price to maturity for all bonds
      (acknowledged limitation: bond floor may be slightly overstated
       for bonds trading near/above conversion price)
    - Rating: All bonds treated as BBB- (lowest BBB rating in universe)
      using the unified BBB spread already in final_discount_rates_daily.csv

Outputs:
    data/clean/bond_floors/
        {TICKER}_floor.csv
    data/clean/bond_floors_all.csv
"""

import pandas as pd
import numpy as np
import os
import warnings
import getpass
warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────────────
USER = getpass.getuser()
if USER == "sarda":
    BASE_DIR = r"C:\Users\sarda\Desktop\cba2"
elif USER == "jinay":
    BASE_DIR = r"C:\Users\jinay\Desktop\cba2"
else:
    BASE_DIR = r"C:\Users\sarda\Desktop\cba2"

STATIC_PATH   = os.path.join(BASE_DIR, "data", "raw", "bond_static", "CONV_BOND_DATA.xlsx")
DISCOUNT_PATH = os.path.join(BASE_DIR, "data", "clean", "final_discount_rates_daily.csv")
BOND_DIR      = os.path.join(BASE_DIR, "data", "raw", "bond_dynamic")
OUTPUT_DIR    = os.path.join(BASE_DIR, "data", "clean", "bond_floors")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Tickers excluded from universe
EXCLUDE_TICKERS = {"F", "HASI", "CSWC"}

# Tenors available in discount table
TENORS = [1, 2, 3, 5, 7, 10]


def load_static():
    """Load static bond file from sheet 'Sheet1'. Ensures 'Ticker' exists (from 'Ticker' or 'Cmn Stk Tkr')."""
    df = pd.read_excel(STATIC_PATH, sheet_name="Sheet1")
    if "Ticker" not in df.columns or df["Ticker"].isna().all():
        if "Cmn Stk Tkr" in df.columns:
            df = df.copy()
            df["Ticker"] = df["Cmn Stk Tkr"]
    return df


# ══════════════════════════════════════════════════════════════════════════
# SECTION 1 — 30/360 DAY COUNT
# ══════════════════════════════════════════════════════════════════════════

def days_30_360(start_date, end_date):
    """
    Calculate year fraction between two dates using 30/360 convention.
    This is the US 30/360 (Bond Basis) method.

    Rule:
        D1 = min(D1, 30)
        If D2 = 31 and D1 >= 30: D2 = 30
        Days = 360*(Y2-Y1) + 30*(M2-M1) + (D2-D1)
    """
    y1, m1, d1 = start_date.year, start_date.month, start_date.day
    y2, m2, d2 = end_date.year,   end_date.month,   end_date.day

    if d1 == 31:
        d1 = 30
    if d2 == 31 and d1 >= 30:
        d2 = 30

    days = 360 * (y2 - y1) + 30 * (m2 - m1) + (d2 - d1)
    return days / 360


# ══════════════════════════════════════════════════════════════════════════
# SECTION 2 — COUPON SCHEDULE GENERATOR
# ══════════════════════════════════════════════════════════════════════════

def generate_coupon_schedule(first_cpn_dt, maturity_dt, cpn_freq="S/A"):
    """
    Generate all coupon payment dates from first coupon to maturity.

    Works backwards from maturity in 6-month steps (for S/A),
    then prepends the first coupon date if not already included.
    This is the standard method — it preserves the stub period
    at the start without creating a stub at the end.

    Parameters:
        first_cpn_dt : first coupon payment date
        maturity_dt  : bond maturity date
        cpn_freq     : coupon frequency ('S/A' = semi-annual, default)

    Returns:
        sorted list of datetime coupon payment dates including maturity
    """
    months_between = 6 if cpn_freq in ["S/A", "SEMI-ANNUAL", "2"] else 12

    # Start from maturity and step backwards
    dates = []
    dt    = maturity_dt

    while dt >= first_cpn_dt:
        dates.append(dt)
        # Step back by months_between months
        month = dt.month - months_between
        year  = dt.year
        while month <= 0:
            month += 12
            year  -= 1
        # Handle end-of-month
        import calendar
        last_day = calendar.monthrange(year, month)[1]
        day      = min(dt.day, last_day)
        dt       = pd.Timestamp(year, month, day)

    # Always include the first coupon date
    if first_cpn_dt not in dates:
        dates.append(first_cpn_dt)

    dates = sorted(set(dates))
    return dates


# ══════════════════════════════════════════════════════════════════════════
# SECTION 3 — DISCOUNT RATE INTERPOLATION
# ══════════════════════════════════════════════════════════════════════════

def get_discount_rate(discount_df, date, maturity_years):
    """
    Interpolate bond floor discount rate (ZC Treasury + BBB spread)
    for a given date and time-to-payment in years.
    """
    date = pd.to_datetime(date)
    row  = discount_df[discount_df["Date"] == date]

    if row.empty:
        idx = (discount_df["Date"] - date).abs().idxmin()
        row = discount_df.iloc[[idx]]

    rates = {t: row[f"discount_{t}y"].values[0] for t in TENORS}

    if maturity_years <= TENORS[0]:
        return rates[TENORS[0]]
    if maturity_years >= TENORS[-1]:
        return rates[TENORS[-1]]

    lower = max(t for t in TENORS if t <= maturity_years)
    upper = min(t for t in TENORS if t >= maturity_years)

    if lower == upper:
        return rates[lower]

    return rates[lower] + (
        (maturity_years - lower) / (upper - lower)
    ) * (rates[upper] - rates[lower])


# ══════════════════════════════════════════════════════════════════════════
# SECTION 4 — BOND FLOOR FOR ONE DATE
# ══════════════════════════════════════════════════════════════════════════

def calculate_bond_floor_on_date(
    pricing_date, coupon_schedule, coupon_amount,
    par, maturity_dt, discount_df, issue_date=None
):
    """
    Calculate the bond floor (clean price as % of par) on a single date.

    Steps:
        1. Filter coupon schedule to future payments only
        2. For each future coupon, calculate t = years to payment (30/360)
        3. Interpolate discount rate r_t for that t
        4. Discount: PV = coupon / (1 + r_t)^t
        5. Also discount par at maturity
        6. Sum all PVs = dirty bond floor
        7. Subtract accrued interest = clean bond floor

    Returns:
        dict with dirty_floor, accrued_interest, clean_floor,
        n_cashflows, effective_discount_rate (all as % of par)
    """
    pricing_date = pd.to_datetime(pricing_date)
    maturity_dt  = pd.to_datetime(maturity_dt)

    if pricing_date >= maturity_dt:
        return None

    # Future coupon dates only
    future_coupons = [d for d in coupon_schedule if d > pricing_date]

    if not future_coupons:
        return None

    total_pv       = 0.0
    rates_used     = []

    # ── Discount each future coupon ────────────────────────────────────────
    for cpn_date in future_coupons:
        t   = days_30_360(pricing_date, cpn_date)
        if t <= 0:
            continue
        r_t = get_discount_rate(discount_df, pricing_date, t)
        pv  = coupon_amount / ((1 + r_t) ** t)
        total_pv   += pv
        rates_used.append(r_t)

    # ── Discount par at maturity ───────────────────────────────────────────
    t_mat = days_30_360(pricing_date, maturity_dt)
    if t_mat > 0:
        r_mat     = get_discount_rate(discount_df, pricing_date, t_mat)
        pv_par    = par / ((1 + r_mat) ** t_mat)
        total_pv += pv_par
        rates_used.append(r_mat)

    dirty_floor_pct = (total_pv / par) * 100   # as % of par (like bond prices)

    # ── Accrued interest ───────────────────────────────────────────────────
    # Find last coupon date (most recent coupon before pricing_date)
    past_coupons = [d for d in coupon_schedule if d <= pricing_date]
    if past_coupons:
        last_cpn_dt   = max(past_coupons)
        # Next coupon date
        future_sorted = sorted(future_coupons)
        next_cpn_dt   = future_sorted[0]

        period_length = days_30_360(last_cpn_dt, next_cpn_dt)
        days_accrued  = days_30_360(last_cpn_dt, pricing_date)

        if period_length > 0:
            accrued_pct = (days_accrued / period_length) * (coupon_amount / par) * 100
        else:
            accrued_pct = 0.0
    else:
        # Before first coupon — follow Bloomberg convention used in exports
        # (no accrued interest before first coupon date)
        accrued_pct = 0.0

    clean_floor_pct = dirty_floor_pct - accrued_pct

    # Effective discount rate = average rate used across all cash flows
    eff_rate = np.mean(rates_used) if rates_used else 0.0

    return {
        "dirty_floor"            : round(dirty_floor_pct, 6),
        "accrued_interest"       : round(accrued_pct, 6),
        "clean_floor"            : round(clean_floor_pct, 6),
        "n_cashflows"            : len(future_coupons) + 1,
        "effective_discount_rate": round(eff_rate * 100, 4),  # in %
    }


# ══════════════════════════════════════════════════════════════════════════
# SECTION 5 — PRICE ONE BOND ACROSS ALL DATES
# ══════════════════════════════════════════════════════════════════════════

def calculate_bond_floor(ticker, static_row, discount_df):
    """
    Calculate daily bond floor for one bond across all its trading dates.
    Uses the bond dynamic file to get the date range.
    """
    # ── Static parameters ──────────────────────────────────────────────────
    par           = float(static_row["Par Amt"].values[0])
    cpn_rate      = float(static_row["Cpn"].values[0]) / 100
    cpn_freq      = str(static_row["Cpn Freq Des"].values[0]).strip()
    first_cpn_dt  = pd.to_datetime(static_row["First Cpn Dt"].values[0])
    maturity_dt   = pd.to_datetime(static_row["Maturity"].values[0])
    issue_dt      = pd.to_datetime(static_row["Issue Date"].values[0])

    # Semi-annual coupon payment amount
    periods_per_year = 2 if cpn_freq in ["S/A", "SEMI-ANNUAL", "2"] else 1
    coupon_amount    = par * cpn_rate / periods_per_year

    print(f"    Par: ${par:,.0f} | Coupon: {cpn_rate*100:.3f}% "
          f"({cpn_freq}) | Payment: ${coupon_amount:.2f}")
    print(f"    First Cpn: {first_cpn_dt.date()} | "
          f"Maturity: {maturity_dt.date()}")

    # ── Generate full coupon schedule ──────────────────────────────────────
    coupon_schedule = generate_coupon_schedule(
        first_cpn_dt, maturity_dt, cpn_freq
    )
    print(f"    Coupon schedule: {len(coupon_schedule)} payments | "
          f"{coupon_schedule[0].date()} to {coupon_schedule[-1].date()}")

    # ── Load bond dynamic file to get trading dates ────────────────────────
    bond_file = os.path.join(BOND_DIR, f"{ticker}_bond.xlsx")
    if not os.path.exists(bond_file):
        raise FileNotFoundError(f"Bond dynamic file not found: {bond_file}")

    bond_df         = pd.read_excel(bond_file, skiprows=5)
    bond_df.columns = ["date", "clean_mid", "dirty_mid"]
    bond_df["date"] = pd.to_datetime(bond_df["date"])
    bond_df         = bond_df.dropna(subset=["date"])
    bond_df         = bond_df.sort_values("date").reset_index(drop=True)

    # ── Calculate bond floor on each trading date ──────────────────────────
    results = []
    skipped = 0

    for _, row in bond_df.iterrows():
        pricing_date = row["date"]
        market_clean = row["clean_mid"]
        market_dirty = row["dirty_mid"]

        result = calculate_bond_floor_on_date(
            pricing_date   = pricing_date,
            coupon_schedule= coupon_schedule,
            coupon_amount  = coupon_amount,
            par            = par,
            maturity_dt    = maturity_dt,
            discount_df    = discount_df,
            issue_date     = issue_dt
        )

        if result is None:
            skipped += 1
            continue

        results.append({
            "date"                    : pricing_date,
            "market_clean_price"      : round(market_clean, 4),
            "market_dirty_price"      : round(market_dirty, 4),
            "clean_floor"             : result["clean_floor"],
            "dirty_floor"             : result["dirty_floor"],
            "accrued_interest"        : result["accrued_interest"],
            "n_cashflows"             : result["n_cashflows"],
            "effective_discount_rate" : result["effective_discount_rate"],
            # Mispricing = market price - bond floor (in price points)
            # Positive = market above floor (normal)
            # Negative = market below floor (theoretical arbitrage signal)
            "premium_to_floor"        : round(market_clean - result["clean_floor"], 4),
        })

    if skipped > 0:
        print(f"    Skipped {skipped} rows (matured / missing)")

    df_out = pd.DataFrame(results)
    return df_out


# ══════════════════════════════════════════════════════════════════════════
# SECTION 6 — RUN ALL BONDS
# ══════════════════════════════════════════════════════════════════════════

def run_all_bonds():
    print("=" * 65)
    print("  BOND FLOOR CALCULATION")
    print("  Method: PV of cash flows discounted at ZC Treasury + BBB spread")
    print("  Day Count: 30/360  |  Calls: Ignored (price to maturity)")
    print("=" * 65)

    static_df   = load_static()
    discount_df = pd.read_csv(DISCOUNT_PATH)
    discount_df["Date"] = pd.to_datetime(discount_df["Date"])

    print(f"\nDiscount table: {len(discount_df)} rows | "
          f"{discount_df['Date'].min().date()} to "
          f"{discount_df['Date'].max().date()}")

    tickers = [
        t for t in static_df["Ticker"].dropna().unique().tolist()
        if t not in EXCLUDE_TICKERS
    ]
    if not tickers:
        sample = static_df["Ticker"].head(5).tolist()
        print(f"\nFound 0 bonds. Ticker column sample: {sample}")
        print("  Check CONV_BOND_DATA.xlsx sheet 'Sheet1': ensure 'Ticker' or 'Cmn Stk Tkr' has ticker symbols.")
        return []
    print(f"\nBonds to price: {len(tickers)} — {tickers}\n")

    all_results = []

    for i, ticker in enumerate(tickers, 1):
        print(f"[{i}/{len(tickers)}] {ticker}...")

        try:
            static_row = static_df[static_df["Ticker"] == ticker]
            result_df  = calculate_bond_floor(ticker, static_row, discount_df)

            if result_df.empty:
                print(f"    WARNING: No results for {ticker}\n")
                continue

            result_df.insert(0, "ticker", ticker)

            # Save individual file
            out_path = os.path.join(OUTPUT_DIR, f"{ticker}_bond_floor.csv")
            result_df.to_csv(out_path, index=False)

            # Summary
            print(f"    Rows: {len(result_df)} | "
                  f"{result_df['date'].min().date()} to "
                  f"{result_df['date'].max().date()}")
            print(f"    Clean floor range: "
                  f"{result_df['clean_floor'].min():.2f} – "
                  f"{result_df['clean_floor'].max():.2f}")
            print(f"    Market price range: "
                  f"{result_df['market_clean_price'].min():.2f} – "
                  f"{result_df['market_clean_price'].max():.2f}")
            print(f"    Premium to floor: "
                  f"{result_df['premium_to_floor'].min():.2f} – "
                  f"{result_df['premium_to_floor'].max():.2f}")
            print(f"    Effective discount rate: "
                  f"{result_df['effective_discount_rate'].min():.3f}% – "
                  f"{result_df['effective_discount_rate'].max():.3f}%")
            print(f"    Saved → bond_floors/{ticker}_floor.csv\n")

            all_results.append(result_df)

        except FileNotFoundError as e:
            print(f"    SKIPPED — {e}\n")
        except Exception as e:
            import traceback
            print(f"    ERROR — {ticker}: {e}")
            traceback.print_exc()
            print()

    # Save combined file
    if all_results:
        combined      = pd.concat(all_results, ignore_index=True)
        combined_path = os.path.join(
            BASE_DIR, "data", "clean", "bond_floors_all.csv"
        )
        combined.to_csv(combined_path, index=False)
        print(f"Combined file saved → data/clean/bond_floors_all.csv")
        print(f"Total rows: {len(combined)} across {len(all_results)} bonds")

    return all_results


# ══════════════════════════════════════════════════════════════════════════
# SECTION 7 — TEST SINGLE BOND
# ══════════════════════════════════════════════════════════════════════════

def test_single_bond(ticker="MTH"):
    """
    Quick test — prints detailed output for one bond and
    compares against market price as a sanity check.
    Run this before running all bonds.
    """
    print("=" * 65)
    print(f"  TEST RUN — {ticker}")
    print("=" * 65)

    static_df   = load_static()
    discount_df = pd.read_csv(DISCOUNT_PATH)
    discount_df["Date"] = pd.to_datetime(discount_df["Date"])

    static_row = static_df[static_df["Ticker"] == ticker]
    result_df  = calculate_bond_floor(ticker, static_row, discount_df)
    result_df.insert(0, "ticker", ticker)

    print(f"\nFirst 5 rows:")
    print(result_df[[
        "date", "market_clean_price", "clean_floor",
        "accrued_interest", "effective_discount_rate", "premium_to_floor"
    ]].head().to_string())

    print(f"\nLast 5 rows:")
    print(result_df[[
        "date", "market_clean_price", "clean_floor",
        "accrued_interest", "effective_discount_rate", "premium_to_floor"
    ]].tail().to_string())

    print(f"\nSummary statistics:")
    print(result_df[[
        "clean_floor", "dirty_floor",
        "accrued_interest", "effective_discount_rate", "premium_to_floor"
    ]].describe().round(4))

    # Sanity checks
    print(f"\nSanity checks:")
    n_below_floor = (result_df["premium_to_floor"] < 0).sum()
    print(f"  Days where market price < bond floor: {n_below_floor} "
          f"({n_below_floor/len(result_df)*100:.1f}%)")
    print(f"  Avg premium to floor: "
          f"{result_df['premium_to_floor'].mean():.2f} price points")
    print(f"  Avg effective discount rate: "
          f"{result_df['effective_discount_rate'].mean():.3f}%")

    # At maturity the floor should approach 100
    print(f"\n  Most recent floor: "
          f"{result_df['clean_floor'].iloc[-1]:.4f}")
    print(f"  Most recent market: "
          f"{result_df['market_clean_price'].iloc[-1]:.4f}")

    return result_df


# ══════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    import sys

    if len(sys.argv) > 1 and sys.argv[1] == "test":
        ticker = sys.argv[2] if len(sys.argv) > 2 else "MTH"
        test_single_bond(ticker)
    else:
        run_all_bonds()

  BOND FLOOR CALCULATION
  Method: PV of cash flows discounted at ZC Treasury + BBB spread
  Day Count: 30/360  |  Calls: Ignored (price to maturity)

Discount table: 1874 rows | 2019-01-02 to 2026-03-09

Bonds to price: 19 — ['KRG', 'PPL1', 'EXC', 'BXP', 'EEFT', 'WEC1', 'DLR', 'WEC2', 'WEC3', 'REXR1', 'REXR2', 'MTH', 'NEE', 'FRT', 'EVRG', 'CDP', 'DUK', 'VTR', 'PPL2']

[1/19] KRG...
    Par: $1,000 | Coupon: 0.750% (S/A) | Payment: $3.75
    First Cpn: 2021-10-01 | Maturity: 2027-04-01
    Coupon schedule: 12 payments | 2021-10-01 to 2027-04-01
    Rows: 1288 | 2021-03-19 to 2026-02-24
    Clean floor range: 79.72 – 96.28
    Market price range: 84.51 – 116.88
    Premium to floor: 1.72 – 26.10
    Effective discount rate: 1.245% – 6.473%
    Saved → bond_floors/KRG_floor.csv

[2/19] PPL1...
    Par: $1,000 | Coupon: 3.000% (S/A) | Payment: $15.00
    First Cpn: 2026-06-01 | Maturity: 2030-12-01
    Coupon schedule: 10 payments | 2026-06-01 to 2030-12-01
    Rows: 53 | 2025-11-20 to 20